# Accessing data from a web API

One of the best things about "web 2.0" is more formal, external facing API's to allow developers to build tools on top of data. These APIs provide a reliable connection to data over a reliable protocol returning a reliable response in an easy, machine-readable format. There are tons of APIs out there. Sometimes you will need to register for a developer key (see the St. Louis Fed's API [here](https://research.stlouisfed.org/docs/api/)) and other times the APIs are free (see Coindesk's API [here](https://www.coindesk.com/api)). ProgrammableWeb has a great [directory](https://www.programmableweb.com/apis/directory).

## With great power comes great responsibility. We should be responsible and polite internet citizens when hitting websites programmatically

Below we will use Coindesk's API to get daily prices for bitcoin. Coindesk's API, like many others, returns a JSON (javascript object notation) object, which we can easily turn into a dataframe. Here we will need the `json` and `requests` libraries.

In [1]:
import datetime
import pandas as pd
import json
import requests

coindeskURL = 'https://api.coindesk.com/v1/bpi/historical/close.json?'

# from API
start = datetime.date(2019, 1 ,1)
end = datetime.date(2019, 7, 1)

url = f'{coindeskURL}start={start:%Y-%m-%d}&end={end:%Y-%m-%d}'

print(f'Hitting this url:{url}')

result = requests.get(url)
result.content

Hitting this url:https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01


ConnectionError: HTTPSConnectionPool(host='api.coindesk.com', port=443): Max retries exceeded with url: /v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x106b6e6f0>: Failed to resolve 'api.coindesk.com' ([Errno 8] nodename nor servname provided, or not known)"))

Digging into this a little bit, there are a few critical components to accessing data. Not all APIs will return the same structure of data. Many times you can look at what is returned by just hitting the URL with your browser. Let's try that

[https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01](https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01)


In this case we get:
```
{
    "bpi":
    {
        "2019-01-01":3869.47,
        "2019-01-02":3941.2167,
        ...
    },
    "disclaimer": "This data was produced from the CoinDesk Bitcoin Price Index.",
    "time":
    {
        "updated":"Jul 2, 2019 00:03:00 UTC",
        "updatedISO":"2019-07-02T00:03:00+00:00"
    }
}
```

`bpi`, `disclaimer`, and `time` are all data that we can reference. The data we're primarily interested in is `bpi`. Different APIs can break up data differently, we will see examples in other APIs later.

Let's take a look at the json result in python.

In [ ]:
jsondata = json.loads(result.content)
jsondata

We can then wrap that up in to a pandas dataframe! Note: pandas does have a `read_json` helper as well, but sometimes it is not well suited given the structure of the json output.

In [ ]:
# note the json gets read with the disclaimer and time outputs as well
data = pd.read_json(result.content)
data

In [ ]:
data = pd.DataFrame({'Bitcoin Price Index': jsondata['bpi']})
data

In [ ]:
%matplotlib inline
data.plot()